# Statistics for Data Analysts/Engineers — Module 8: t-Tests

In Module 7 we built the whole hypothesis-testing toolkit using a proportion as our
example. Now we'll apply it to the most common question in practice: **do means differ
from each other?**. The **t-test** family handles three variants of this question: one
sample against a reference value, two independent groups, and paired data (the same
subject measured twice).

## Table of contents
1. [One-sample t-test](#sec1)
2. [Two-sample independent t-test](#sec2)
3. [Checking assumptions: normality and equal variance](#sec3)
4. [Paired t-test](#sec4)
5. [Effect size: Cohen's d](#sec5)
6. [Summary and exercises](#sec6)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. One-sample t-test

The **one-sample t-test** checks whether a sample mean differs from a specific reference
value. It's exactly the same idea as the proportion test from Module 7, just for a mean
instead of a rate. Let's check: does mean monthly customer spend differ from $200?

- H0: `mu = 200`
- H1: `mu != 200`

In [ ]:
from scipy import stats

t_statistic, p_value = stats.ttest_1samp(customers["monthly_spend"], popmean=200)
print(f"t = {t_statistic:.3f}")
print(f"p-value = {p_value:.4f}")

alpha = 0.05
print("Reject H0" if p_value <= alpha else "Fail to reject H0")


> 📊 **t instead of z, because the population's sigma is unknown**
>
> `ttest_1samp` uses the t-distribution (not z) for exactly the same reason as the confidence intervals in Module 6 -- we don't know the population's true standard deviation, so we estimate it from the sample, which introduces extra uncertainty.

<a id="sec2"></a>
## 2. Two-sample independent t-test

The **independent (two-sample) t-test** compares the means of TWO independent groups.
Let's check: does mean monthly spend differ between campaign group A and group B?

- H0: `mu_A = mu_B`
- H1: `mu_A != mu_B`

In [ ]:
spend_a = customers[customers["campaign_group"] == "A"]["monthly_spend"]
spend_b = customers[customers["campaign_group"] == "B"]["monthly_spend"]

t_statistic2, p_value2 = stats.ttest_ind(spend_a, spend_b)
print(f"Mean A: {spend_a.mean():.2f}, Mean B: {spend_b.mean():.2f}")
print(f"t = {t_statistic2:.3f}, p-value = {p_value2:.4f}")


This result shouldn't surprise you: recall that campaign group affects `converted` in
our data, but NOT `monthly_spend` -- so expect a large p-value (no grounds to reject H0).
Important lesson: **a significant effect on one variable (conversion) doesn't mean a
significant effect on another (spend)**.

<a id="sec3"></a>
## 3. Checking assumptions: normality and equal variance

The t-test (in its classic form) assumes an approximately normal distribution of data in
each group (or a sample large enough for the CLT from Module 5 to kick in), and -- in the
"Student's" version -- equal variances between the two groups. Two quick checks:

- **Shapiro-Wilk** (`stats.shapiro`) — a normality test. H0: the data comes from a normal
  distribution.
- **Levene** (`stats.levene`) — an equal-variance test. H0: the variances are equal.

In [ ]:
# normality in each group separately
print("Shapiro-Wilk (group A):", stats.shapiro(spend_a))
print("Shapiro-Wilk (group B):", stats.shapiro(spend_b))

# equality of variances
print("Levene:", stats.levene(spend_a, spend_b))


> ⚠️ **Remember the right-skewed spend from Module 2? It matters here**
>
> `monthly_spend` is clearly right-skewed (Module 2), so the Shapiro-Wilk test will most likely reject normality. Is that a problem? Not necessarily -- thanks to the CLT (Module 5), the distribution of the **sample MEAN** approaches normal anyway at our `n` (hundreds of observations per group), even if the raw data doesn't. The t-test is fairly robust to moderate departures from normality at large sample sizes -- it's more of a concern for very small samples (`n < 15-20`).

By default `scipy.stats.ttest_ind` assumes EQUAL variances (the classic Student's
test). When Levene indicates unequal variances, a better choice is **Welch's test**
(`equal_var=False`) -- it doesn't assume equal variances and is a safe default choice
when you're not sure:

In [ ]:
welch_statistic, welch_p = stats.ttest_ind(spend_a, spend_b, equal_var=False)
print(f"Welch's test: t = {welch_statistic:.3f}, p-value = {welch_p:.4f}")


> 📊 **When in doubt, use Welch's test**
>
> Many statisticians recommend defaulting to `equal_var=False` almost always -- when the variances really are equal, Welch's test gives nearly identical results to the classic t-test, and when they aren't equal, Welch's test is much safer. The cost of this caution is very low.

<a id="sec4"></a>
## 4. Paired t-test

The **paired t-test** compares TWO measurements ON THE SAME subjects (e.g. before/after
an intervention) -- rather than two independent groups of different people. Our
`spend_before`/`spend_after` columns (spend of the same 400 customers before and after a
loyalty-program rollout) are a textbook example of paired data.

In [ ]:
paired_statistic, paired_p = stats.ttest_rel(customers["spend_after"], customers["spend_before"])
print(f"Mean before: {customers['spend_before'].mean():.2f}")
print(f"Mean after: {customers['spend_after'].mean():.2f}")
print(f"t = {paired_statistic:.3f}, p-value = {paired_p:.6f}")


> ⚠️ **Don't confuse a paired test with an independent one!**
>
> If someone (incorrectly) used `ttest_ind(spend_before, spend_after)` instead of `ttest_rel`, they'd lose the information that these are the SAME people measured twice -- an independent test ignores the pairing and is therefore less sensitive (it loses power). Rule of thumb: if two measurements are on the same units, you almost always want a paired test.

In [ ]:
# a paired test is fundamentally a one-sample t-test on the before/after DIFFERENCES
differences = customers["spend_after"] - customers["spend_before"]
alt_statistic, alt_p = stats.ttest_1samp(differences, popmean=0)
print(f"t (on differences) = {alt_statistic:.3f}, p-value = {alt_p:.6f}")


Same result as `ttest_rel` -- because it's exactly the same question: "does the mean
`after - before` difference differ significantly from zero?".

<a id="sec5"></a>
## 5. Effect size: Cohen's d

The p-value tells you whether an effect is **statistically significant**, but NOT whether
it's practically **large or small** (remember the warning from Module 7!). **Cohen's d**
is a popular effect-size measure for a mean difference -- the mean difference expressed in
units of standard deviation:

```
d = (mean1 - mean2) / pooled_std
```

Conventional interpretation thresholds (Cohen, 1988): `0.2` = small effect, `0.5` =
medium, `0.8` = large -- but that's just a convention; context always matters.

In [ ]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    s1, s2 = group1.std(ddof=1), group2.std(ddof=1)
    pooled_s = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
    return (group1.mean() - group2.mean()) / pooled_s


d = cohens_d(customers["spend_after"], customers["spend_before"])
print(f"Cohen's d (after vs before): {d:.3f}")


Notice the paired test from section 4 gave a TINY p-value (a strongly statistically
significant effect) -- check whether `d` actually indicates a large effect, or just that
we had a lot of data and a small, but very consistent, increase.

<a id="sec6"></a>
## 6. Summary and exercises

In this module we covered:
- the one-sample t-test (`ttest_1samp`) -- comparing a mean to a reference value,
- the independent two-sample t-test (`ttest_ind`) -- and the difference between
  Student's and Welch's versions,
- checking assumptions: normality (Shapiro-Wilk) and equal variance (Levene),
- the paired t-test (`ttest_rel`) -- and why pairing matters,
- Cohen's d as a measure of effect size, independent of the p-value.

> 📝 **Exercise 1: A one-sample test for age**
>
> Use a one-sample t-test to check whether mean customer age differs significantly from `45` years. State H0/H1 and interpret the result at `alpha=0.05`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
t_age, p_age = stats.ttest_1samp(customers["age"], popmean=45)
print(f"t={t_age:.3f}, p={p_age:.4f}")
print("Reject H0" if p_age <= 0.05 else "Fail to reject H0")
```
</details>

> 📝 **Exercise 2: Spend by city (two selected)**
>
> Use Welch's test (`equal_var=False`) to compare mean `monthly_spend` between New York and Chicago. Also check Levene's test for these two groups before running the test.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
spend_newyork = customers[customers["city"] == "New York"]["monthly_spend"]
spend_chicago = customers[customers["city"] == "Chicago"]["monthly_spend"]

print("Levene:", stats.levene(spend_newyork, spend_chicago))
t_cities, p_cities = stats.ttest_ind(spend_newyork, spend_chicago, equal_var=False)
print(f"t={t_cities:.3f}, p={p_cities:.4f}")
```
</details>

> 📝 **Exercise 3: Cohen's d for spend by group**
>
> Compute Cohen's d for `monthly_spend` between group A and group B (use the `cohens_d` function from section 5). Compare it with the t-test result from section 2 -- is a small/large effect consistent with whether the test came out significant?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
d_groups = cohens_d(spend_a, spend_b)
print(f"Cohen's d: {d_groups:.3f}")
```

Since the t-test from section 2 found no significance, expect a very small |d| (close to 0) -- a small d and a non-significant result paint a consistent picture of no real difference.
</details>

> 📝 **Exercise 4: A paired test on a subsample**
>
> Draw a subsample of 20 customers (`customers.sample(n=20, random_state=5)`) and repeat the paired test `spend_after` vs `spend_before` ONLY on that subsample. Is the result still statistically significant with such a small sample?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
subsample = customers.sample(n=20, random_state=5)
t_small, p_small = stats.ttest_rel(subsample["spend_after"], subsample["spend_before"])
print(f"t={t_small:.3f}, p={p_small:.6f}")
```

The effect is strong and consistent enough (the loyalty program raises spend for nearly every customer) that it will likely remain significant even at n=20 -- but the p-value will typically be larger than with the full n=400 sample (less data = less evidence, as in Module 7).
</details>

> 🔥 **Challenge: a simulation-based check on the t-test**
>
> Verify the INDEPENDENT two-sample t-test with a permutation method (similar to the p-value simulation in Module 7): pool `spend_a` and `spend_b` together, shuffle that pool randomly 10,000 times and split it into two groups of the same sizes as the original A/B, computing the mean difference each time. Compute the fraction of permutations where |permuted difference| >= |original difference| -- that's your permutation p-value. Compare it with the p-value from `ttest_ind` in section 2.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
import numpy as np

rng = np.random.default_rng(3)
pooled = np.concatenate([spend_a, spend_b])
n_a = len(spend_a)
original_diff = abs(spend_a.mean() - spend_b.mean())

num_permutations = 10_000
extreme_count = 0
for _ in range(num_permutations):
    shuffled = rng.permutation(pooled)
    sim_group_a = shuffled[:n_a]
    sim_group_b = shuffled[n_a:]
    sim_diff = abs(sim_group_a.mean() - sim_group_b.mean())
    if sim_diff >= original_diff:
        extreme_count += 1

permutation_p = extreme_count / num_permutations
print(f"P-value (permutation): {permutation_p:.4f}")
print(f"P-value (ttest_ind, section 2): {p_value2:.4f}")
```

The permutation method assumes NOTHING about the shape of the distribution (unlike the t-test) -- the fact that it gives a very similar result is additional confirmation that the t-test is working correctly here despite the data's skewness.
</details>

## What's next?

T-tests compare the MEANS of continuous variables. In **Module 9** we'll tackle questions
about **categorical** variables: whether two categorical features are related to each
other (a chi-square test of independence) and whether an observed category distribution
matches an expected one (a chi-square goodness-of-fit test).